<div style="background: linear-gradient(135deg, #1e3a5f 0%, #2d5a87 100%); padding: 26px; border-radius: 10px; margin: 30px 0 18px 0;">
<h1 style="color: white; margin: 0; font-size: 2em;">Semana 1 · Práctica</h1>
<p style="color: #a8d4ff; margin: 8px 0 0 0; font-size: 1.05em;">Un extractor con LangChain y LangGraph que corre solo</p>
</div>
<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Qué vas a hacer</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">Vas a armar, paso a paso, un proceso que lee balances y guarda cuatro datos de cada uno, sin que nadie lo esté mirando.</p>
<ul style="margin: 0 0 12px 0; padding-left: 24px;">
<li style="margin-bottom: 6px;">Usás <strong>LangChain</strong> para llamar al modelo y <strong>LangGraph</strong> para el workflow.</li>
<li style="margin-bottom: 6px;">Los datos son <strong>otros cuatro campos del balance</strong>, que no vimos en clase: efectivo, activo corriente, pasivo total y patrimonio.</li>
<li style="margin-bottom: 6px;">El schema, el prompt y las correcciones los escribís vos. Nadie los vio todavía, así que los errores que aparezcan van a ser los tuyos.</li>
<li style="margin-bottom: 6px;">Se corrige con el mismo método de la clase: se compara contra XBRL.</li>
</ul>
<table style="border-collapse: collapse; margin: 6px 0 14px 0; font-size: 0.95em; width: 100%;">
<thead>
<tr>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;"></th>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;"></th>
</tr>
</thead>
<tbody>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><strong>Costo de API</strong></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">cerca de USD 1. Estas celdas llaman al modelo en vivo y necesitan tu API key</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><strong>Empresas</strong></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">13. Las de la clase menos Exxon, porque su recorte no trae el balance</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><strong>Autochequeo</strong></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">la última celda dice qué pasos están bien. No hay nada que entregar</td>
</tr>
</tbody>
</table>
<p style="margin: 0 0 12px 0;">Si algo de la clase no te quedó claro, <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">week_1_repaso.ipynb</code> es la clase con partes para completar.</p></div></div>

In [2]:
# Setup: corré esta celda primero.
import json
import os
import subprocess
import sys
import time

sys.path.insert(0, "..")  # la raíz del repo es la raíz de los imports

from typing import Optional, TypedDict

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel, Field

from week_1.lab import data, evals, validate
from week_1.lab.reference import TO_MILLIONS, Unit

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("GEMINI_API_KEY"), "Falta la API key: creá un archivo .env con GEMINI_API_KEY=..."

meli = data.load_filing("MELI")
print("Empresas de la práctica:", ", ".join(data.tickers("tarea")))

Empresas de la práctica: AAPL, MSFT, GOOG, AMZN, META, TSLA, JPM, JNJ, WMT, MELI, YPF, GLOB, IRS


<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Los cuatro campos</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><table style="border-collapse: collapse; margin: 6px 0 14px 0; font-size: 0.95em; width: 100%;">
<thead>
<tr>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;">Campo del schema</th>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;">Qué es</th>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;">Atención</th>
</tr>
</thead>
<tbody>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">cash_millions</code></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">efectivo y equivalentes, del balance</td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">en un banco la línea se llama distinto</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">current_assets_millions</code></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">total del activo corriente</td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">hay empresas que no separan corriente y no corriente</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">total_liabilities_millions</code></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">total del pasivo</td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">hay balances que no tienen esa línea</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">equity_millions</code></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">patrimonio que corresponde a los accionistas de la empresa</td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">hay balances con dos totales de patrimonio</td>
</tr>
</tbody>
</table>
<ul style="margin: 0 0 12px 0; padding-left: 24px;">
<li style="margin-bottom: 6px;">Todos van en <strong>millones</strong> de la moneda del balance.</li>
<li style="margin-bottom: 6px;">Si el balance no tiene la línea, lo correcto es <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">null</code>. No hay que calcularla.</li>
<li style="margin-bottom: 6px;">La respuesta correcta es el valor que la empresa etiquetó en XBRL, igual que en clase.</li>
<li style="margin-bottom: 6px;">Los nombres de los campos tienen que ser esos cuatro, porque son los que busca el eval.</li>
</ul></div></div>

<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Qué es LangChain</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">LangChain es una librería de Python, de código abierto, para escribir programas que usan modelos de lenguaje. En la clase llamamos a OpenAI con su propio cliente (<code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">client.responses.parse</code>). LangChain pone una capa encima de esa llamada, igual para todos los proveedores.</p>
<p style="margin: 0 0 12px 0;">Lo que vas a usar en esta práctica:</p>
<table style="border-collapse: collapse; margin: 6px 0 14px 0; font-size: 0.95em; width: 100%;">
<thead>
<tr>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;">Pieza</th>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;">Para qué sirve</th>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;">En la clase lo hicimos con</th>
</tr>
</thead>
<tbody>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">ChatOpenAI</code></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">el modelo. Cambiar de proveedor es cambiar esta clase por <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">ChatAnthropic</code> o <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">ChatGoogleGenerativeAI</code></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">client.responses.create(model=...)</code></td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">ChatPromptTemplate</code></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">un prompt con variables y con roles (<em>system</em>, <em>user</em>)</td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">una lista de diccionarios armada a mano</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">.with_structured_output(Schema)</code></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">que la respuesta cumpla un schema de Pydantic</td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">client.responses.parse(text_format=Schema)</code></td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">.with_retry()</code></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">reintentar si la API falla</td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">no lo hicimos</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">prompt \| modelo</code></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">unir las piezas en una cadena que se ejecuta con <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">.invoke()</code></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">llamar una función después de otra</td>
</tr>
</tbody>
</table>
<ul style="margin: 0 0 12px 0; padding-left: 24px;">
<li style="margin-bottom: 6px;">Se instala con <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">pip install langchain langchain-openai</code>. Ya está en <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">requirements-w1.txt</code>.</li>
<li style="margin-bottom: 6px;">La ventaja es que mucho código repetido ya viene escrito, y que casi todas las herramientas del ecosistema (trazas, evaluación, bases vectoriales) se conectan con LangChain.</li>
<li style="margin-bottom: 6px;">La desventaja es que toma decisiones por vos y no siempre se ven. En el paso 3 aparece una.</li>
</ul></div></div>
<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Qué es LangGraph</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">LangGraph es otra librería del mismo equipo. Sirve para escribir el <strong>workflow</strong>: qué paso va después de cuál, cuándo se vuelve atrás y cuándo se termina.</p>
<ul style="margin: 0 0 12px 0; padding-left: 24px;">
<li style="margin-bottom: 6px;">Una cadena de LangChain va siempre en línea recta: prompt, modelo, salida.</li>
<li style="margin-bottom: 6px;">Nuestro proceso no es una línea recta. Después de validar, a veces hay que volver a extraer, y a veces hay que mandar el documento a una persona. En la clase eso fue un <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">for</code> con varios <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">if</code> dentro de <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">extract_with_repair</code>.</li>
<li style="margin-bottom: 6px;">En LangGraph ese recorrido se escribe como un <strong>grafo</strong>. Cada paso es un <strong>nodo</strong>, que es una función de Python. Las flechas entre nodos son las <strong>aristas</strong>. Todos los nodos leen y escriben un mismo diccionario, el <strong>estado</strong>.</li>
<li style="margin-bottom: 6px;">A cambio de escribirlo así, LangGraph puede mostrar cada paso mientras corre, dibujar el grafo, guardar el estado para retomar una corrida que se cortó, y frenar en un nodo hasta que una persona apruebe.</li>
<li style="margin-bottom: 6px;">Es la librería con la que se arman agentes en las semanas que siguen. Un agente es un grafo donde el que elige la próxima flecha es el modelo.</li>
</ul>
<p style="margin: 0 0 12px 0;">Las dos se usan juntas, pero son independientes. Dentro de un nodo de LangGraph puede haber una cadena de LangChain, una llamada directa a OpenAI o código sin ningún modelo.</p>
<table style="border-collapse: collapse; margin: 6px 0 14px 0; font-size: 0.95em; width: 100%;">
<thead>
<tr>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;"></th>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;">LangChain</th>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;">LangGraph</th>
</tr>
</thead>
<tbody>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">Resuelve</td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">una llamada al modelo</td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">el recorrido entre varias llamadas</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">Pasos de esta práctica</td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">1 a 6</td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">7 y 8</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">Documentación</td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://docs.langchain.com/oss/python/langchain/overview">docs.langchain.com · LangChain</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://docs.langchain.com/oss/python/langgraph/overview">docs.langchain.com · LangGraph</a></td>
</tr>
</tbody>
</table></div></div>

<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Paso 1: comprobar que LangChain funciona</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">ChatOpenAI</code> es el modelo visto desde LangChain. Se crea con el nombre del modelo y se llama con <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">.invoke(...)</code>.</p>
<p style="margin: 0 0 12px 0;">Creá el modelo con <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">model="gpt-5.4-mini"</code> y pedile que conteste "ok". La respuesta es un objeto: el texto está en <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">.content</code> y los tokens en <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">.usage_metadata</code>.</p></div></div>

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

# TODO 1: creá el modelo (En mi caso con GEMINI)
modelo = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.2
)

agent= create_agent(
    model= modelo,
    system_prompt="You are a helpful assistant",
)

result= agent.invoke(
    {"messages": [{"role": "user", "content": "Responde solo: ok"}]}
)
print(result["messages"][-1].content)

respuesta = modelo.invoke("Respondé solo: ok")
print(respuesta.content)
print(respuesta.usage_metadata)

c:\Users\t3tri\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text', 'text': 'ok', 'extras': {'signature': 'EmAKXgFpFH0TZOPeUaxM+8xezP4E6OS35mJTgywSeLsSwQ9IIZO9C8zi7GJDcVlaRH7fYefBgq+2JzwdNR8nMJStWlZi8fRoEdwof0p2l/qKN2ZhicRYOxpLHzHqaaQLJFc='}}]


c:\Users\t3tri\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'ok', 'extras': {'signature': 'EmAKXgFpFH0TsSAT3WVyAnSWyz2ZOOrWUynI3C9e7mc2UpvDE6LPsc95QSQWGpYkyJ0+mzEbn0Uxwn/EW91plzT/IouFfuqIIt71Omp9NAdZhaA7GyOeriw1zYLJRGnCKWQ='}}]
{'input_tokens': 6, 'output_tokens': 1, 'total_tokens': 7, 'input_token_details': {'cache_read': 0}}


In [4]:
# ✅ CHECK 1
contenido= respuesta.content

if isinstance(contenido,list):
    texto_respuesta = "".join([c if isinstance(c,str) else str(c) for c in contenido])
else:
    texto_respuesta = str(contenido)
assert "ok" in texto_respuesta, "TODO 1: modelo = ChatGoogleGenerativeAI(model='gemini-3.5-flash-lite')"
assert respuesta.usage_metadata["input_tokens"] > 0
print("OK. LangChain llega al modelo.")

OK. LangChain llega al modelo.


<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Paso 2: un prompt con variables y una cadena</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><ul style="margin: 0 0 12px 0; padding-left: 24px;">
<li style="margin-bottom: 6px;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">ChatPromptTemplate</code> es un prompt con variables. Le ponemos dos mensajes: uno <em>system</em> con las instrucciones y uno <em>user</em> con el documento.</li>
<li style="margin-bottom: 6px;">Las piezas se unen con <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">|</code>. <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">prompt | modelo</code> es una <strong>cadena</strong>, y se ejecuta con <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">.invoke({...})</code>, pasando un valor para cada variable.</li>
</ul>
<p style="margin: 0 0 12px 0;">Armá la cadena y preguntale cuánto efectivo tenía MercadoLibre al cierre del último ejercicio. Por ahora la respuesta es texto libre.</p></div></div>

In [5]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "{instrucciones}"),
    ("user", "{documento}"),
])

# TODO 2: uní el prompt con el modelo
cadena = prompt | modelo

r = cadena.invoke({
    "instrucciones": "Decí cuánto efectivo (cash and cash equivalents) tenía la empresa al cierre del último ejercicio. Una oración.",
    "documento": meli,
})

if isinstance(r.content,list):
    r.content= "" .join([c if isinstance(c,str) else str(c) for c in r.content])
print(r.content)

c:\Users\t3tri\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'type': 'text', 'text': 'Al cierre del ejercicio terminado el 31 de diciembre de 2025, la empresa tenía $3,670 millones en efectivo y equivalentes de efectivo (*cash and cash equivalents*).', 'extras': {'signature': 'EmAKXgFpFH0TWLMnK/xY429jgUSORggcGUBP76XJ4ghlaSMa7J+vuRMMU2NXZfaQurOI03o9Bzf0hU+pVsbXrNfwBcs8FRrDc4ApZNoMCxt9HCo4IqZsZplacK/L8U3IPjg='}}


In [6]:
# ✅ CHECK 2: el efectivo de MercadoLibre es 3.670 millones
assert any(n in r.content for n in ("3,670", "3.670", "3670")), "TODO 2: cadena = prompt | modelo"
print("OK. La cadena lee el balance.")

OK. La cadena lee el balance.


<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Paso 3: la salida con un schema</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">Texto libre no sirve para guardar en una tabla. <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">.with_structured_output(Schema)</code> le pide al modelo una salida que cumpla un schema de Pydantic.</p>
<p style="margin: 0 0 12px 0;">Escribí <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">BalanceV1</code> con los cuatro campos de la tabla de arriba, todos <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">Optional[float]</code>, cada uno con su <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">description</code>. Para esta primera versión alcanza con descripciones cortas.</p>
<p style="margin: 0 0 12px 0;">Dos parámetros que dejamos puestos:</p>
<ul style="margin: 0 0 12px 0; padding-left: 24px;">
<li style="margin-bottom: 6px;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">method="json_schema"</code> manda el schema como formato de respuesta, igual que en clase. Si no se indica, LangChain usa otro mecanismo (<em>function calling</em>) y los resultados cambian un poco.</li>
<li style="margin-bottom: 6px;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">include_raw=True</code> hace que la salida sea un diccionario con dos cosas: <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">"parsed"</code>, el objeto con los datos, y <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">"raw"</code>, el mensaje original, de donde salen los tokens.</li>
</ul></div></div>

In [7]:
class BalanceV1(BaseModel):
    # TODO 3: los cuatro campos, Optional[float], con una description corta cada uno
    cash_millions: Optional[float]= Field(description= "Cash and cash equivalents expressed in millions")
    current_assets_millions: Optional[float]= Field(description="Total current assets expressed in millions")
    total_liabilities_millions: Optional[float]= Field(description="Total liabilities expressed in millions")
    equity_millions: Optional[float]= Field(description="Total net worth expressed in millions")


PROMPT_V1 = "Extract the balance sheet data from the SEC filing below. If not found, leave as null."

cadena_v1 = prompt | modelo.with_structured_output(BalanceV1, method="json_schema", include_raw=True)

salida = cadena_v1.invoke({"instrucciones": PROMPT_V1, "documento": meli})
print(salida["parsed"].model_dump_json(indent=2))
print(salida["raw"].usage_metadata)

c:\Users\t3tri\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{
  "cash_millions": 3670.0,
  "current_assets_millions": 33573.0,
  "total_liabilities_millions": 35919.0,
  "equity_millions": 6748.0
}
{'input_tokens': 16043, 'output_tokens': 65, 'total_tokens': 16108, 'input_token_details': {'cache_read': 0}}


In [8]:
# ✅ CHECK 3
esperados = {"cash_millions", "current_assets_millions", "total_liabilities_millions", "equity_millions"}
assert esperados <= set(BalanceV1.model_fields), f"TODO 3: faltan campos: {esperados - set(BalanceV1.model_fields)}"
assert isinstance(salida["parsed"], BalanceV1)
print("OK. La salida es un objeto con los cuatro campos.")

OK. La salida es un objeto con los cuatro campos.


<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Paso 4: medir</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">Un resultado que parece razonable en una empresa no dice nada de las otras doce. Hay que medir.</p>
<p style="margin: 0 0 12px 0;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">evals.run_eval</code> es el eval de la clase. Acepta cualquier función <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">extractor(ticker, texto)</code> que devuelva dos cosas: el objeto con los cuatro campos y el mensaje original de LangChain (<code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">salida["raw"]</code>), de donde saca los tokens para calcular el costo.</p>
<p style="margin: 0 0 12px 0;">Completá el extractor y corré el eval con <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">suite="tarea"</code>. Son 13 empresas por 4 campos, <strong>52 comparaciones</strong>. Cuesta cerca de USD 0,16.</p></div></div>

In [9]:
def extractor_v1(ticker, texto):
    # TODO 4: llamá a cadena_v1 con PROMPT_V1 y el texto
    salida = cadena_v1.invoke({"instrucciones": PROMPT_V1, "documento": texto})
    parsed_data= salida.get("parsed")
    raw_data= salida.get("raw")

    return parsed_data, raw_data

v1 = evals.run_eval(extractor=extractor_v1, model="gemini-3.5-flash-lite", label="tarea-v1", suite="tarea")
print(v1)
v1.by_field()

c:\Users\t3tri\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\t3tri\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\t3tri\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\t3tri\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3

[tarea-v1] accuracy 45/52 = 86.5% | modelo gemini-3.5-flash-lite | costo de lista $0.000 (pagado ahora: $0.000) | 234,359 tokens in | 0s de modelo


,ok,total,accuracy
field,,,
cash_millions,13,13,1.00
current_assets_millions,13,13,1.00
equity_millions,8,13,0.62
total_liabilities_millions,11,13,0.85


In [10]:
# ✅ CHECK 4
assert len(v1.df) == 52, "TODO 4: el eval tiene que comparar 52 valores (13 empresas por 4 campos)"
aciertos_v1 = int(v1.df["ok"].sum())
print(f"OK. Tu punto de partida: {aciertos_v1} de 52.")

OK. Tu punto de partida: 45 de 52.


<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #4b5563; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;"><span style="font-size: 0.72em; font-weight: 600; letter-spacing: 0.04em; text-transform: uppercase; opacity: 0.85; display: block; margin-bottom: 2px;">👀 Deberías ver</span></div><div style="background: #f3f4f6; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">Con un schema y un prompt cortos, entre 30 y 40 de 52. El resto de la práctica es subir ese número.</p></div></div>

<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Paso 5: mirar los errores</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">Antes de tocar el prompt hay que entender qué pasó. Son las reglas de la clase: una oración por error, sin categorías, sin soluciones, mirando el documento.</p>
<ul style="margin: 0 0 12px 0; padding-left: 24px;">
<li style="margin-bottom: 6px;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">v1.failures()</code> muestra cada error con el valor esperado, el devuelto y una explicación posible.</li>
<li style="margin-bottom: 6px;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">data.find_in_filing(ticker, "número")</code> busca un número en el balance y muestra lo que hay alrededor.</li>
</ul>
<p style="margin: 0 0 12px 0;">Anotá en <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">mis_notas</code> al menos cuatro errores. Después agrupalos: ¿cuántos tipos distintos de error hay?</p></div></div>

In [11]:
v1.failures()

,ticker,field,expected,got,diagnosis
0,AMZN,total_liabilities_millions,NaN,406977.000,la empresa NO reporta este dato: se esperaba null y el modelo devolvió un valor (¿lo calculó? ¿lo inventó?)
1,TSLA,equity_millions,82137.000,82807.000,es el patrimonio TOTAL (incluye participación no controlante)
2,WMT,total_liabilities_millions,NaN,178781.000,la empresa NO reporta este dato: se esperaba null y el modelo devolvió un valor (¿lo calculó? ¿lo inventó?)
3,WMT,equity_millions,99617.000,105887.000,es el patrimonio TOTAL (incluye participación no controlante)
4,YPF,equity_millions,10814.000,11044.000,es el patrimonio TOTAL (incluye participación no controlante)
5,GLOB,equity_millions,2093.601,2161.636,es el patrimonio TOTAL (incluye participación no controlante)
6,IRS,equity_millions,1577804.000,1671967.000,es el patrimonio TOTAL (incluye participación no controlante)


In [12]:
# Para abrir un caso. Cambiá la empresa y el número por los de tu tabla de errores
data.find_in_filing("WMT", "178781.000", width=400)

mis_notas = {
    # TODO 5: (ticker, campo): "qué pasó, en una oración"
    ("AMZN","total_liabilities_millions"): "la empresa NO reporta este dato: se esperaba null y el modelo devolvió un valor (¿lo calculó? ¿lo inventó?)",
    ("WMT","total_liabilities_millions"): "la empresa NO reporta este dato: se esperaba null y el modelo devolvió un valor (¿lo calculó? ¿lo inventó?)",
    ("WMT","equity_millions"): "es el patrimonio TOTAL (incluye participación no controlante)",
    ("GLOB","equity_millions"):"es el patrimonio TOTAL (incluye participación no controlante)"
}

WMT: 0 apariciones de '178781.000'


In [13]:
# ✅ CHECK 5: no necesita API key
assert len(mis_notas) >= 4, f"TODO 5: anotá al menos 4 errores (llevás {len(mis_notas)})"
for clave, nota in mis_notas.items():
    assert len(nota.split()) >= 8, f"{clave}: contá qué pasó en el balance, con una oración completa"
print(f"OK. {len(mis_notas)} errores descriptos.")

OK. 4 errores descriptos.


<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Paso 6: mejorar y volver a medir</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">Con tus errores a la vista, escribí la versión 2. Tres cosas que suelen hacer falta, por lo que vimos en clase:</p>
<ol style="margin: 0 0 12px 0; padding-left: 24px;">
<li style="margin-bottom: 6px;"><strong>Definir cada campo sin ambigüedad</strong>, en su <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">description</code>. Qué línea del balance es, de qué año, y cuándo va <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">null</code>.</li>
<li style="margin-bottom: 6px;"><strong>No pedirle cuentas al modelo.</strong> Que copie el importe tal como está impreso y que informe aparte la unidad del balance (<code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">statement_unit</code>). La conversión a millones la hace una función tuya, <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">convertir_a_millones</code>.</li>
<li style="margin-bottom: 6px;"><strong>Un prompt con las seis partes</strong> de la tabla de la clase.</li>
</ol>
<p style="margin: 0 0 12px 0;">El schema que lee el modelo (<code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">BalanceV2</code>) tiene los importes como están impresos. El que mira el eval tiene que tener los cuatro campos en millones: para eso está <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">BalanceEnMillones</code>, que ya viene escrito.</p>
<p style="margin: 0 0 12px 0;">Corré el eval completo cada vez que cambies algo. <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">evals.compare()</code> guarda todas tus corridas.</p></div></div>

In [14]:
class BalanceV2(BaseModel):
    statement_unit: Unit = Field(description="The unit in which the financial statements are expressed (e.g., in thousands, in millions, whole units)")   # TODO 6a: qué tiene que informar acá el modelo
    # TODO 6a: los cuatro importes, tal como están impresos, con una description precisa cada uno:
    # cash_reported, current_assets_reported, total_liabilities_reported, equity_reported
    cash_reported: Optional[float] = Field(
        default=None,
        description="Cash and cash equivalents at the end of the period")
    current_assets_reported: Optional[float] = Field(
        default=None,
        description="Total current assets at the end of the period")
    total_liabilities_reported: Optional[float] = Field(
        default=None,
        description="Total liabilities at the end of the period")
    equity_reported: Optional[float] = Field(
        default=None,
        description="Total stockholders' equity / Total equity")

class BalanceEnMillones(BaseModel):
    cash_millions: Optional[float] = None
    current_assets_millions: Optional[float] = None
    total_liabilities_millions: Optional[float] = None
    equity_millions: Optional[float] = None


# TODO 6b: tu prompt
PROMPT_V2 = """Act as a senior financial auditor expert in data extraction from SEC reports (10-K / 10-Q).
Analyze the provided financial document and extract the following data from the Balance Sheet as of the end of the last reported period with maximum precision:

1. statement_unit: The monetary scale in which the figures are reported.
2. cash_reported: Cash and cash equivalents.
3. current_assets_reported: Total current assets.
4. total_liabilities_reported: Total liabilities.
5. equity_reported: Total shareholders' equity.

Strict rules:
- If a field is not explicitly reported or cannot be determined with certainty, leave it as null (None).
- Do not invent or approximate values; extract the exact number as printed on the corresponding financial line item.
- Do not fill in the corresponding financial line item with external values.
- Do not assume values ​​for the corresponding financial line item.
- Do not calculate incorrect values ​​for the corresponding financial line item."""

def convertir_a_millones(b: BalanceV2) -> BalanceEnMillones:
    # TODO 6c: multiplicá cada importe por TO_MILLIONS[b.statement_unit.value]. Un importe vacío sigue vacío.
    factor= TO_MILLIONS[b.statement_unit.value] if b.statement_unit else 1.0

    return BalanceEnMillones(
        cash_millions=b.cash_reported * factor if b.cash_reported is not None else None,
        current_assets_millions=b.current_assets_reported * factor if b.current_assets_reported is not None else None,
        total_liabilities_millions=b.total_liabilities_reported * factor if b.total_liabilities_reported is not None else None,
        equity_millions=b.equity_reported * factor if b.equity_reported is not None else None,
    )



cadena_v2 = prompt | modelo.with_structured_output(BalanceV2, method="json_schema", include_raw=True)


def extractor_v2(ticker, texto):
    salida = cadena_v2.invoke({"instrucciones": PROMPT_V2, "documento": texto})
    return convertir_a_millones(salida["parsed"]), salida["raw"]


v2 = evals.run_eval(extractor=extractor_v2, model="gemini-3.5-flash-lite", label="tarea-v2", suite="tarea")
print(v2)
v2.failures()

c:\Users\t3tri\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\t3tri\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\t3tri\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\t3tri\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3

[tarea-v2] accuracy 46/52 = 88.5% | modelo gemini-3.5-flash-lite | costo de lista $0.000 (pagado ahora: $0.000) | 236,881 tokens in | 0s de modelo


,ticker,field,expected,got,diagnosis
0,AMZN,total_liabilities_millions,NaN,406977.000,la empresa NO reporta este dato: se esperaba null y el modelo devolvió un valor (¿lo calculó? ¿lo inventó?)
1,WMT,total_liabilities_millions,NaN,178778.000,la empresa NO reporta este dato: se esperaba null y el modelo devolvió un valor (¿lo calculó? ¿lo inventó?)
2,WMT,equity_millions,99617.000,105887.000,es el patrimonio TOTAL (incluye participación no controlante)
3,YPF,equity_millions,10814.000,11044.000,es el patrimonio TOTAL (incluye participación no controlante)
4,GLOB,equity_millions,2093.601,2161.636,es el patrimonio TOTAL (incluye participación no controlante)
5,IRS,equity_millions,1577804.000,1671967.000,es el patrimonio TOTAL (incluye participación no controlante)


In [15]:
# ✅ CHECK 6
aciertos_v2 = int(v2.df["ok"].sum())
print(f"v1: {aciertos_v1} de 52   →   v2: {aciertos_v2} de 52")
assert aciertos_v2 >= 46, "Todavía no. Mirá v2.failures(), corregí y volvé a correr. Hacen falta 46 de 52 para seguir."
assert aciertos_v2 > aciertos_v1, "v2 tiene que acertar más que v1"
print("OK. Podés seguir.")

v1: 45 de 52   →   v2: 46 de 52
OK. Podés seguir.


<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Paso 7: el workflow en LangGraph</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">En producción no vas a tener la respuesta correcta para comparar. Hace falta lo que vimos en v4: pedir la cita de cada importe, validar con reglas y reintentar. Y lo que no pasa las reglas no se guarda: va a una lista para que lo revise una persona.</p>
<p style="margin: 0 0 12px 0;">Ese recorrido tiene un lazo, y LangGraph sirve para escribirlo como un grafo:</p>
<table style="border-collapse: collapse; margin: 6px 0 14px 0; font-size: 0.95em; width: 100%;">
<thead>
<tr>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;">Concepto</th>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;">Qué es</th>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;">En esta práctica</th>
</tr>
</thead>
<tbody>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><strong>Estado</strong></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">un diccionario tipado que pasa de nodo en nodo</td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">el documento, la extracción, los errores, los intentos</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><strong>Nodo</strong></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">una función que recibe el estado y devuelve solo los campos que cambia</td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">extraer</code>, <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">validar</code>, <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">guardar</code>, <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">a_revision</code></td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><strong>Arista</strong></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">qué nodo sigue</td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">de <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">extraer</code> siempre se va a <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">validar</code></td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><strong>Arista condicional</strong></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">una función que mira el estado y devuelve el nombre del próximo nodo</td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">después de <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">validar</code>: guardar, reintentar o mandar a revisión</td>
</tr>
</tbody>
</table>
<pre style="background: #f1f5f9; color: #0f172a; border: 1px solid #cbd5e1; padding: 14px 16px; border-radius: 8px; overflow-x: auto; font-size: 0.88em; line-height: 1.45; margin: 6px 0 14px 0;"><code style="background: transparent; color: #0f172a; padding: 0;"> START ──► extraer ──► validar ──┬─ pasa las reglas ───────────────► guardar ────► END
              ▲                  ├─ no pasa y quedan intentos ──┐
              └──────────────────┘◄─────────────────────────────┘
                                 └─ no pasa y no quedan intentos ─► a_revision ──► END
</code></pre>
<p style="margin: 0 0 12px 0;">Primero, un grafo de juguete con esa forma. No llama a ningún modelo: el "modelo" es una función que la primera vez lee mal el separador de miles.</p></div></div>
<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #15803d; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;"><span style="font-size: 0.72em; font-weight: 600; letter-spacing: 0.04em; text-transform: uppercase; opacity: 0.85; display: block; margin-bottom: 2px;">⌨️ El código que sigue</span>Un grafo de juguete, sin modelo</div><div style="background: #f0fdf4; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><ul style="margin: 0 0 12px 0; padding-left: 24px;">
<li style="margin-bottom: 6px;"><strong>Línea 4:</strong> el estado que viaja de nodo en nodo.</li>
<li style="margin-bottom: 6px;"><strong>Línea 24:</strong> la arista condicional. Mira los errores y los intentos y devuelve el nombre del próximo nodo.</li>
<li style="margin-bottom: 6px;"><strong>Línea 37:</strong> conecta <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">validar</code> con los tres destinos posibles.</li>
<li style="margin-bottom: 6px;"><strong>Línea 44:</strong> corre el grafo y muestra qué devuelve cada nodo. No llama a ningún modelo.</li>
<li style="margin-bottom: 6px;"><strong>Mirar:</strong> <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">extraer</code> aparece dos veces en la salida. Esa es la vuelta del lazo.</li>
</ul></div></div>

In [16]:
from langgraph.graph import END, START, StateGraph


class EstadoJuguete(TypedDict):
    texto: str
    numero: Optional[float]
    errores: list[str]
    intentos: int


def extraer_juguete(estado):
    # Un "modelo" de juguete: la primera vez lee 3,670 como un decimal
    numero = 3.670 if estado["intentos"] == 0 else 3670.0
    return {"numero": numero, "intentos": estado["intentos"] + 1}


def validar_juguete(estado):
    como_en_el_texto = f"{estado['numero']:,.0f}"
    if como_en_el_texto in estado["texto"]:
        return {"errores": []}
    return {"errores": [f"{estado['numero']} no figura en el texto"]}


def decidir_juguete(estado):
    if not estado["errores"]:
        return "guardar"
    return "extraer" if estado["intentos"] < 3 else "a_revision"


juguete = StateGraph(EstadoJuguete)
juguete.add_node("extraer", extraer_juguete)
juguete.add_node("validar", validar_juguete)
juguete.add_node("guardar", lambda estado: {})
juguete.add_node("a_revision", lambda estado: {})
juguete.add_edge(START, "extraer")
juguete.add_edge("extraer", "validar")
juguete.add_conditional_edges("validar", decidir_juguete, ["guardar", "extraer", "a_revision"])
juguete.add_edge("guardar", END)
juguete.add_edge("a_revision", END)
grafo_juguete = juguete.compile()

# stream() muestra qué devuelve cada nodo, en orden
inicial = {"texto": "Cash and cash equivalents 3,670 2,635", "numero": None, "errores": [], "intentos": 0}
for paso in grafo_juguete.stream(inicial):
    print(paso)

{'extraer': {'numero': 3.67, 'intentos': 1}}
{'validar': {'errores': ['3.67 no figura en el texto']}}
{'extraer': {'numero': 3670.0, 'intentos': 2}}
{'validar': {'errores': []}}
{'guardar': None}


<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #4b5563; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;"><span style="font-size: 0.72em; font-weight: 600; letter-spacing: 0.04em; text-transform: uppercase; opacity: 0.85; display: block; margin-bottom: 2px;">👀 Deberías ver</span></div><div style="background: #f3f4f6; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">extraer</code> devuelve 3.67, <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">validar</code> encuentra un error, vuelve a <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">extraer</code>, que ahora devuelve 3670.0, <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">validar</code> no encuentra errores y termina en <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">guardar</code>.</p></div></div>
<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Ahora el tuyo</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><ol style="margin: 0 0 12px 0; padding-left: 24px;">
<li style="margin-bottom: 6px;"><strong>El schema.</strong> <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">Balance</code> es tu <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">BalanceV2</code> más una cita por cada importe: <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">cash_quote</code>, <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">current_assets_quote</code>, <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">total_liabilities_quote</code>, <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">equity_quote</code>, todos <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">Optional[str]</code>. En la <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">description</code>, pedí la línea exacta del balance, copiada tal cual.</li>
<li style="margin-bottom: 6px;"><strong>Las reglas.</strong> Ya vienen armadas abajo, con las funciones de <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">validate</code> que vimos en clase.</li>
<li style="margin-bottom: 6px;"><strong>La cadena.</strong> Igual que antes, con un mensaje más en el prompt: <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">("placeholder", "{correcciones}")</code>. Es el lugar donde van, en un reintento, la respuesta anterior del modelo y los errores.</li>
<li style="margin-bottom: 6px;"><strong>Los cuatro nodos y la arista condicional.</strong></li>
</ol>
<p style="margin: 0 0 12px 0;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">.with_retry()</code> de LangChain reintenta cuando la llamada falla por la red o por la API. No reintenta cuando el dato está mal. Ese reintento es el lazo de tu grafo.</p></div></div>

In [18]:
class Balance(BalanceV2):
    # TODO 7a: las cuatro citas, Optional[str], cada una con su description
    cash_quote: Optional[str] = Field(
        default=None,
        description="Exact text (quote) from the document where the net worth was found")
    current_assets_quote: Optional[str] = Field(
        default=None,
        description="Exact text (quote) from the document where the net worth was found")
    total_liabilities_quote: Optional[str] = Field(
        default=None,
        description="Exact text (quote) from the document where the net worth was found")
    equity_quote: Optional[str] = Field(
        default=None,
        description="Exact text (quote) from the document where the net worth was found")


CAMPOS = ("cash", "current_assets", "total_liabilities", "equity")
REGLAS = ([validate.required_rule("cash_reported"), validate.required_rule("equity_reported")]
          + [validate.evidence_rule(f"{c}_reported", f"{c}_quote") for c in CAMPOS])
MAX_INTENTOS = 3
# USD por millón de tokens (entrada, salida). Verificar en https://openai.com/api/pricing
PRECIOS = {
    "gemini-3.5-flash-lite": (0.075, 0.30),  # Tarifas aproximadas o ajustadas para flash-lite
    "gemini-2.5-flash": (0.075, 0.30),
    "gemini-2.5-pro": (1.25, 5.00),
}


def costo_usd(model, uso):
    entrada, salida = PRECIOS.get(model, (0.0, 0.0))
    return (uso["input_tokens"] * entrada + uso["output_tokens"] * salida) / 1e6


class Estado(TypedDict):
    ticker: str
    documento: str
    extraccion: Optional[Balance]
    respuesta_cruda: str
    errores: list[str]
    intentos: int
    costo_usd: float


def construir_grafo(model, store):
    prompt_con_correcciones = ChatPromptTemplate.from_messages([
        ("system", "{instrucciones}"),
        ("user", "{documento}"),
        ("placeholder", "{correcciones}"),
    ])
    cadena = prompt_con_correcciones | ChatGoogleGenerativeAI(model=model).with_structured_output(
        Balance, method="json_schema", include_raw=True
    ).with_retry(stop_after_attempt=3)

    def extraer(estado):
        correcciones = []
        if estado["errores"]:
            # TODO 7b: dos mensajes. ("assistant", la respuesta anterior) y ("user", los errores, pidiendo que corrija)
            correcciones = [
                ("assistant", estado["respuesta_cruda"]),
                ("user","Tu extracción anterior tuvo los siguientes errores o faltas de evidencia:\n" + "\n".join(estado["errores"]) + "\nPor favor, corrige estos errores basándote estrictamente en el documento")
            ]
        salida = cadena.invoke({"instrucciones": PROMPT_V2, "documento": estado["documento"],
                                "correcciones": correcciones})
        return {
            "extraccion": salida["parsed"],
            "respuesta_cruda": json.dumps(salida["parsed"].model_dump(mode="json")),
            "intentos": estado["intentos"] + 1,
            "costo_usd": estado["costo_usd"] + costo_usd(model, salida["raw"].usage_metadata),
        }

    def validar(estado):
        # TODO 7c: devolvé {"errores": ...} usando validate.run_rules(extraccion, documento, REGLAS)
        errores_detectados= validate.run_rules(estado["extraccion"], estado["documento"], REGLAS)
        return {"errores": errores_detectados}

    def decidir(estado):
        # TODO 7d: devolvé "guardar", "extraer" o "a_revision"
        if not estado["errores"]:
            return "guardar"
        elif estado["intentos"]>= MAX_INTENTOS:
            return "a_revision"
        else:
            return "extraer"

    def guardar(estado):
        store.save({"ticker": estado["ticker"], **convertir_a_millones(estado["extraccion"]).model_dump(),
                    "intentos": estado["intentos"]})
        return {}

    def a_revision(estado):
        store.revision.append({"ticker": estado["ticker"], "errores": estado["errores"]})
        return {}

    grafo = StateGraph(Estado)
    # TODO 7e: los cuatro nodos, las aristas y la arista condicional. Mirá el grafo de juguete.
    grafo.add_node("extraer", extraer)
    grafo.add_node("validar", validar)
    grafo.add_node("guardar", guardar)
    grafo.add_node("a_revision", a_revision)

    grafo.set_entry_point("extraer") #inicio del grado


    grafo.add_edge("extraer", "validar") #Extraigo y Valido

    grafo.add_conditional_edges( #Una vez validado decido el destino del estado
        "validar", decidir,
        {
            "extraer": "extraer",
            "guardar": "guardar",
            "a_revision": "a_revision"
        }
    )

    grafo.add_edge("guardar", END)
    grafo.add_edge("a_revision", END)

    return grafo.compile()


class StoreDePrueba:
    # Un lugar donde guardar mientras probamos: dos listas en memoria
    def __init__(self):
        self.guardados, self.revision = [], []

    def save(self, registro):
        self.guardados.append(registro)


store_de_prueba = StoreDePrueba()
grafo = construir_grafo("gemini-3.5-flash-lite", store_de_prueba)

for ticker in ("MELI", "JPM"):
    final = grafo.invoke({"ticker": ticker, "documento": data.load_filing(ticker), "extraccion": None,
                          "respuesta_cruda": "", "errores": [], "intentos": 0, "costo_usd": 0.0})
    print(ticker, "| intentos:", final["intentos"], "| errores:", final["errores"], f"| USD {final['costo_usd']:.4f}")

pd.DataFrame(store_de_prueba.guardados)

MELI | intentos: 1 | errores: [] | USD 0.0013
JPM | intentos: 1 | errores: [] | USD 0.0015


,ticker,cash_millions,current_assets_millions,total_liabilities_millions,equity_millions,intentos
0,MELI,3670.0,33573.0,35919.0,6748.0,1
1,JPM,343338.0,NaN,4062462.0,362438.0,1


In [19]:
# ✅ CHECK 7
citas = {"cash_quote", "current_assets_quote", "total_liabilities_quote", "equity_quote"}
assert citas <= set(Balance.model_fields), f"TODO 7a: faltan citas: {citas - set(Balance.model_fields)}"
procesados = {r["ticker"] for r in store_de_prueba.guardados} | {r["ticker"] for r in store_de_prueba.revision}
assert procesados == {"MELI", "JPM"}, "TODO 7e: cada documento tiene que terminar en guardar o en a_revision"
guardado = {r["ticker"]: r for r in store_de_prueba.guardados}
if "MELI" in guardado:
    assert abs(guardado["MELI"]["cash_millions"] - 3670) < 20, "El efectivo de MercadoLibre es 3.670 millones"
if "JPM" in guardado:
    assert guardado["JPM"]["current_assets_millions"] is None, "JPMorgan no informa activo corriente: va None"
print("OK. El grafo extrae, valida y decide.")

OK. El grafo extrae, valida y decide.


<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Paso 8: que corra solo</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">Hasta acá todo corrió en un notebook, con vos mirando. Ahora lo pasás a un script que se pueda ejecutar desde la terminal o desde un programador de tareas.</p>
<ol style="margin: 0 0 12px 0; padding-left: 24px;">
<li style="margin-bottom: 6px;">Abrí <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">week_1/produccion/pipeline.py</code>. Ya tiene los imports, las rutas de los archivos y el <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">main</code>.</li>
<li style="margin-bottom: 6px;">Pegá tu <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">Balance</code>, tu prompt, <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">convertir_a_millones</code> y <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">construir_grafo</code> en los lugares marcados.</li>
<li style="margin-bottom: 6px;">Escribí las dos funciones que faltan:
   - <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">extractor_para_eval</code>, que conecta tu grafo con el eval.
   - <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">correr</code>, que procesa la lista de empresas sin ayuda de nadie. Tiene que saltear lo que ya está guardado y lo que espera revisión, seguir si un documento falla, y dejar un reporte.</li>
<li style="margin-bottom: 6px;">Probalo desde <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">week_1/</code>:</li>
</ol>
<pre style="background: #f1f5f9; color: #0f172a; border: 1px solid #cbd5e1; padding: 14px 16px; border-radius: 8px; overflow-x: auto; font-size: 0.88em; line-height: 1.45; margin: 6px 0 14px 0;"><code style="background: transparent; color: #0f172a; padding: 0;">python produccion/pipeline.py --model gpt-5.4-nano --reset   # procesa las 13 empresas
python produccion/pipeline.py --model gpt-5.4-nano           # segunda corrida: no tiene que procesar nada
python produccion/check_tarea.py --rapido               # los tres controles, con nano
python produccion/check_tarea.py                        # la versión completa, con mini
</code></pre>
<p style="margin: 0 0 12px 0;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">check_tarea.py</code> hace tres controles: que acierte 48 de 52 con <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">mini</code>, que la segunda corrida no repita trabajo, y que una empresa que no existe quede anotada sin frenar a las demás.</p>
<p style="margin: 0 0 12px 0;">Para dejarlo corriendo todas las noches a las 3, con <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">cron</code>:</p>
<pre style="background: #f1f5f9; color: #0f172a; border: 1px solid #cbd5e1; padding: 14px 16px; border-radius: 8px; overflow-x: auto; font-size: 0.88em; line-height: 1.45; margin: 6px 0 14px 0;"><code style="background: transparent; color: #0f172a; padding: 0;">0 3 * * *  cd /ruta/al/repo/week_1 &amp;&amp; ../.venv/bin/python produccion/pipeline.py &gt;&gt; pipeline.log 2&gt;&amp;1
</code></pre>
<p style="margin: 0 0 12px 0;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">produccion/pipeline_solucion.py</code> es una solución posible. Conviene mirarla después de intentarlo.</p></div></div>

In [1]:
# Dejalo vacío para revisar tu produccion/pipeline.py
ARGUMENTOS = []

In [2]:
# ✅ CHECK 8: corre los tres controles. Tarda unos 3 minutos y cuesta cerca de USD 0,30.
controles = subprocess.run([sys.executable, "produccion/check_tarea.py", *ARGUMENTOS], capture_output=True, text=True)
print(controles.stdout[-2500:] or controles.stderr[-1500:])

assert controles.returncode == 0, "Todavía no pasa los tres controles. Arriba dice cuál falta."
print("OK.")

NameError: name 'subprocess' is not defined

<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Autochequeo</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">La celda de abajo revisa los ocho pasos y dice cuáles están bien y cuáles faltan. No hay nada que entregar. Se puede correr en cualquier momento, aunque no hayas terminado.</p></div></div>

In [ ]:
# ✅ Autochequeo: no llama al modelo, mira lo que ya corriste
g = globals()
campos = {"cash_millions", "current_assets_millions", "total_liabilities_millions", "equity_millions"}
citas = {"cash_quote", "current_assets_quote", "total_liabilities_quote", "equity_quote"}
pasos = [
    ("1. LangChain llega al modelo", "respuesta" in g and "ok" in str(g["respuesta"].content).lower()),
    ("2. La cadena lee el balance", "r" in g and any(n in str(getattr(g["r"], "content", "")) for n in ("3,670", "3.670", "3670"))),
    ("3. El schema tiene los cuatro campos", "BalanceV1" in g and campos <= set(getattr(g["BalanceV1"], "model_fields", {}))),
    ("4. Mediste la primera versión", "v1" in g and len(g["v1"].df) == 52),
    ("5. Describiste al menos cuatro errores", len(g.get("mis_notas", {})) >= 4),
    ("6. La segunda versión acierta 46 de 52 o más", g.get("aciertos_v2", 0) >= 46),
    ("7. El grafo guarda o manda a revisión", "Balance" in g and citas <= set(getattr(g["Balance"], "model_fields", {}))
        and "store_de_prueba" in g and len(g["store_de_prueba"].guardados) + len(g["store_de_prueba"].revision) >= 2),
    ("8. El script pasa los tres controles", getattr(g.get("controles"), "returncode", 1) == 0),
]
for nombre, ok in pasos:
    print("✅" if ok else "⬜", nombre)
print()
if "aciertos_v1" in g and "aciertos_v2" in g:
    print(f"Aciertos: v1 {g['aciertos_v1']} de 52, v2 {g['aciertos_v2']} de 52")
hechos = sum(ok for _, ok in pasos)
print(f"{hechos} de {len(pasos)} pasos completos." + (" Terminaste la práctica." if hechos == len(pasos) else ""))

<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Recursos</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><table style="border-collapse: collapse; margin: 6px 0 14px 0; font-size: 0.95em; width: 100%;">
<thead>
<tr>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;">Tema</th>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;">Sitio</th>
</tr>
</thead>
<tbody>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://docs.langchain.com/oss/python/langchain/overview">LangChain, visión general</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">docs.langchain.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://docs.langchain.com/oss/python/langchain/structured-output">Salida estructurada en LangChain</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">docs.langchain.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://docs.langchain.com/oss/python/langgraph/overview">LangGraph, visión general</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">docs.langchain.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://docs.langchain.com/oss/python/langgraph/quickstart">LangGraph, primeros pasos</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">docs.langchain.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://docs.langchain.com/oss/python/langgraph/graph-api">La API de grafos: estado, nodos, aristas</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">docs.langchain.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://docs.langchain.com/oss/python/langgraph/workflows-agents">Workflows y agentes con LangGraph, con un ejemplo de cada patrón</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">docs.langchain.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://docs.langchain.com/oss/python/langgraph/persistence">Guardar el estado entre pasos (<em>checkpoints</em>)</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">docs.langchain.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://docs.langchain.com/oss/python/langgraph/interrupts">Frenar el grafo para que apruebe una persona (<em>interrupts</em>)</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">docs.langchain.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://docs.langchain.com/langsmith/home">LangSmith, para ver la traza de cada corrida</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">docs.langchain.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://academy.langchain.com/courses/intro-to-langgraph">Curso gratuito: Introduction to LangGraph, de LangChain Academy</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">academy.langchain.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://github.com/langchain-ai/langgraph">Código fuente y ejemplos</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">github.com</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><a href="https://www.anthropic.com/engineering/building-effective-agents">Lectura: Building effective agents, de Anthropic</a></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">www.anthropic.com</td>
</tr>
</tbody>
</table>
<p style="margin: 0 0 12px 0;">Versiones con las que se probó esta práctica: <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">langchain 1.4.2</code>, <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">langchain-openai 1.6.3</code>, <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">langgraph 1.2.12</code>. Están en <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">requirements-w1.txt</code>.</p></div></div>

<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Para terminar: el proceso corriendo, y cómo llevarlo a una máquina virtual</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">Tu script ya no necesita el notebook ni a vos. Falta ponerlo en una máquina que esté siempre prendida. Alcanza con la máquina virtual más chica de cualquier proveedor (AWS EC2, Google Compute Engine, DigitalOcean), con 1 CPU y 1 GB de memoria: el trabajo pesado lo hace la API de OpenAI, la máquina solo manda texto y guarda resultados.</p>
<p style="margin: 0 0 12px 0;">Los pasos, en una VM con Ubuntu:</p>
<pre style="background: #f1f5f9; color: #0f172a; border: 1px solid #cbd5e1; padding: 14px 16px; border-radius: 8px; overflow-x: auto; font-size: 0.88em; line-height: 1.45; margin: 6px 0 14px 0;"><code style="background: transparent; color: #0f172a; padding: 0;"># 1. Entrar a la máquina
ssh ubuntu@IP_DE_LA_VM

# 2. Instalar git y Python (3.10 o más nuevo)
sudo apt update &amp;&amp; sudo apt install -y git python3-venv

# 3. Bajar el código e instalar las dependencias
git clone https://github.com/juanwisz/curso-agentes.git
cd curso-agentes
python3 -m venv .venv
.venv/bin/pip install -r requirements-w1.txt

# 4. La API key, en un archivo que solo lee tu usuario. No va en el repo.
echo 'OPENAI_API_KEY=sk-...' &gt; .env
chmod 600 .env

# 5. Probarlo una vez a mano
cd week_1
../.venv/bin/python produccion/pipeline.py

# 6. Programarlo: crontab -e, y agregar esta línea (todos los días a las 3)
0 3 * * *  cd /home/ubuntu/curso-agentes/week_1 &amp;&amp; ../.venv/bin/python produccion/pipeline.py &gt;&gt; pipeline.log 2&gt;&amp;1
</code></pre>
<ul style="margin: 0 0 12px 0; padding-left: 24px;">
<li style="margin-bottom: 6px;">La hora de <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">cron</code> es la de la VM, que suele estar en UTC. Las 3 UTC son las 0 de Argentina.</li>
<li style="margin-bottom: 6px;">Cada noche el script procesa solo lo que no vio antes. Por eso importaba que la segunda corrida no repitiera trabajo.</li>
</ul>
<p style="margin: 0 0 12px 0;">Qué mirar al día siguiente:</p>
<table style="border-collapse: collapse; margin: 6px 0 14px 0; font-size: 0.95em; width: 100%;">
<thead>
<tr>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;">Archivo</th>
<th style="background: #1e3a5f; color: #ffffff; text-align: left; padding: 8px 12px; border: 1px solid #1e3a5f;">Qué tiene</th>
</tr>
</thead>
<tbody>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">pipeline.log</code></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">lo que imprimió cada corrida</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">artifacts/produccion_reporte.json</code></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">guardados, a revisión, errores, costo y duración de la última corrida</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">artifacts/produccion_store.jsonl</code></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">los datos extraídos</td>
</tr>
<tr>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;"><code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">artifacts/produccion_revision.jsonl</code></td>
<td style="padding: 8px 12px; border: 1px solid #d0d7de; vertical-align: top; color: #1f2937; background: #ffffff; overflow-wrap: anywhere;">lo que no pasó las reglas. Es la lista de trabajo de una persona</td>
</tr>
</tbody>
</table></div></div>
<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Cómo correrlo sobre todos los balances de la SEC</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">En la práctica el script lee 13 balances que ya están en el repo. Para procesar todo lo que se presenta ante la SEC cambia de dónde salen los documentos. La extracción, las reglas y el grafo quedan igual.</p>
<ol style="margin: 0 0 12px 0; padding-left: 24px;">
<li style="margin-bottom: 6px;"><strong>La lista de empresas.</strong> La SEC publica todas las empresas con su identificador (CIK) en <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">https://www.sec.gov/files/company_tickers.json</code>. Son cerca de 10.000.</li>
<li style="margin-bottom: 6px;"><strong>Los informes de cada una.</strong> <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">https://data.sec.gov/submissions/CIK0001099590.json</code> trae los informes de una empresa (ese CIK es el de MercadoLibre), con el tipo (10-K, 20-F), la fecha y el nombre del archivo. El script se queda con los 10-K y 20-F que todavía no procesó. Es lo que hace <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">week_1/components/sec_edgar.py</code> en este repo, y lo que usa la clase con <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">DESCARGAR_DE_EDGAR = True</code>.</li>
<li style="margin-bottom: 6px;"><strong>Las reglas de la SEC.</strong> Cada pedido tiene que llevar un encabezado <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">User-Agent</code> con tu nombre y tu mail, y no se pueden hacer más de 10 pedidos por segundo.</li>
<li style="margin-bottom: 6px;"><strong>Recortar el documento.</strong> Un 10-K entero tiene cientos de páginas. Antes de llamar al modelo hay que quedarse con la parte de los estados contables, como hicimos con el recorte de 70.000 caracteres de la clase.</li>
<li style="margin-bottom: 6px;"><strong>Procesar en paralelo.</strong> De a uno, 7.000 informes tardan unas tres horas. LangGraph procesa varios a la vez con <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">grafo.batch(lista_de_estados, config={"max_concurrency": 8})</code>. Si la API responde que se pasó el límite de pedidos, <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">.with_retry()</code> espera y reintenta.</li>
</ol>
<pre style="background: #f1f5f9; color: #0f172a; border: 1px solid #cbd5e1; padding: 14px 16px; border-radius: 8px; overflow-x: auto; font-size: 0.88em; line-height: 1.45; margin: 6px 0 14px 0;"><code style="background: transparent; color: #0f172a; padding: 0;">import requests

SEC = {&quot;User-Agent&quot;: &quot;Tu Nombre tu@mail.com&quot;}
empresas = requests.get(&quot;https://www.sec.gov/files/company_tickers.json&quot;, headers=SEC).json()

for empresa in empresas.values():
    cik = str(empresa[&quot;cik_str&quot;]).zfill(10)
    informes = requests.get(f&quot;https://data.sec.gov/submissions/CIK{cik}.json&quot;, headers=SEC).json()[&quot;filings&quot;][&quot;recent&quot;]
    # quedarse con los 10-K y 20-F nuevos, bajarlos, recortarlos y pasarlos por el grafo
</code></pre>
<p style="margin: 0 0 12px 0;">Cuánto cuesta, con los números de tu corrida: 13 balances con <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">gpt-5.4-mini</code> salieron cerca de USD 0,18, o sea 1,4 centavos cada uno. Por año se presentan unos 7.000 informes anuales, que serían cerca de USD 100. Con <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">gpt-5.4-nano</code> es la cuarta parte, con más documentos en revisión. Si no hay apuro, la <em>Batch API</em> de OpenAI cobra la mitad y devuelve los resultados dentro de las 24 horas.</p>
<p style="margin: 0 0 12px 0;">Un punto para tener en cuenta: estos cuatro campos ya están en XBRL, y en producción se leerían de ahí, gratis y sin errores. Los usamos en la práctica porque permiten corregir. Este mismo workflow se justifica para lo que no está etiquetado: las notas, los factores de riesgo, los contratos, o los documentos de tu empresa.</p></div></div>
<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">Lo que le falta para ser un sistema real</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><ul style="margin: 0 0 12px 0; padding-left: 24px;">
<li style="margin-bottom: 6px;"><strong>Avisos.</strong> El script termina con código 1 si hubo errores. Falta que alguien se entere: un mail de <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">cron</code> o un mensaje a un canal.</li>
<li style="margin-bottom: 6px;"><strong>Una base de datos</strong> en lugar de archivos, como la opción <code style="background: #e2e8f0; color: #0f172a; padding: 1px 6px; border-radius: 4px; font-size: 0.92em;">USE_MONGO</code> de la clase.</li>
<li style="margin-bottom: 6px;"><strong>La key</strong> en un gestor de secretos del proveedor, y un límite de gasto mensual configurado en la cuenta de OpenAI.</li>
</ul></div></div>
<div style="border: 1px solid #d0d7de; border-radius: 10px; overflow: hidden; margin: 14px 0; box-shadow: 0 1px 4px rgba(15, 23, 42, 0.10); max-width: 1000px;"><div style="background: #1e3a5f; color: #ffffff; padding: 12px 22px; font-size: 1.3em; font-weight: 700; line-height: 1.3;">El proceso corriendo</div><div style="background: #ffffff; color: #1f2937; padding: 16px 22px 6px 22px; font-size: 1.04em; line-height: 1.55;"><p style="margin: 0 0 12px 0;">La celda de abajo corre tu script desde cero, igual que lo va a correr la VM, y muestra lo que dejó guardado. Cuesta cerca de USD 0,20.</p></div></div>

In [ ]:
# El proceso corriendo: el mismo comando que va a ejecutar cron
script = "produccion/pipeline_solucion.py" if "--solucion" in ARGUMENTOS else "produccion/pipeline.py"
corrida = subprocess.run([sys.executable, script, "--reset"], capture_output=True, text=True)
print(corrida.stdout or corrida.stderr[-1500:])

print(open("artifacts/produccion_reporte.json", encoding="utf-8").read())
pd.read_json("artifacts/produccion_store.jsonl", lines=True)